In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/fka/prompts.chat/prompts.csv")

df.head()

,act,prompt,for_devs,type,contributor
0,Ethereum Developer,Imagine you are an experienced Ethereum develo...,True,TEXT,ameya-2003
1,Linux Terminal,I want you to act as a linux terminal. I will ...,True,TEXT,f
2,English Translator and Improver,"I want you to act as an English translator, sp...",False,TEXT,f
3,Job Interviewer,I want you to act as an interviewer. I will be...,False,TEXT,"f,iltekin"
4,JavaScript Console,I want you to act as a javascript console. I w...,True,TEXT,omerimzali


In [ ]:
# LoRA (Low-Rank Adaptation) — это метод эффективного дообучения больших языковых моделей (LLM), который позволяет адаптировать модель к новым задачам или доменам, не изменяя исходные веса модели. Вместо того чтобы обновлять все параметры модели, LoRA добавляет небольшие обучаемые матрицы низкого ранга (адаптеры) в определённые слои модели. Это значительно снижает количество обучаемых параметров и уменьшает требования к вычислительным ресурсам.
# QLoRA (Quantized LoRA) — это расширение метода LoRA, которое сочетает в себе преимущества квантования и адаптации. В QLoRA базовая модель квантуется в 4-битный формат (NF4), что уменьшает её размер и ускоряет инференс, а обучаемые адаптеры LoRA остаются в FP16, что позволяет эффективно дообучать модели с большим количеством параметров на ограниченных ресурсах, таких как одна бытовая GPU.
# 1. Установка необходимых библиотек
# !pip install -q -U transformers peft trl datasets bitsandbytes accelerate

import json
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# ==========================================
# 2. Подготовка кастомного датасета (JSONL)
# ==========================================
custom_data = [
    {
        "messages": [
            {"role": "system", "content": "Ты IT-консультант по архитектуре LLM."},
            {"role": "user", "content": "Что лучше выбрать для RAG: Sparse или Dense эмбеддинги?"},
            {"role": "assistant", "content": "Лучше всего использовать гибридный поиск (Hybrid Search): Dense (например, BGE) отвечает за семантический смысл, а Sparse (BM25) точно находит ключевые слова, артикулы и имена."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Ты IT-консультант по архитектуре LLM."},
            {"role": "user", "content": "Какое главное преимущество GQA перед MHA?"},
            {"role": "assistant", "content": "Grouped-Query Attention (GQA) сокращает размер KV-Cache в памяти GPU в 4-8 раз, сохраняя качество генерации на уровне стандартного Multi-Head Attention."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Ты IT-консультант по архитектуре LLM."},
            {"role": "user", "content": "Зачем нужен QLoRA?"},
            {"role": "assistant", "content": "QLoRA квантует веса базовой модели в 4-бит (NF4) и добавляет обущаемые адаптеры LoRA в FP16, что позволяет дообучать модели на 7B+ параметров на одной бытовой GPU."}
        ]
    }
]

# Записываем обучающую выборку в локальный файл train.jsonl
with open("train.jsonl", "w", encoding="utf-8") as f:
    for item in custom_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Загружаем датасет через библиотеку datasets
dataset = load_dataset("json", data_files="train.jsonl", split="train")

# ==========================================
# 3. Загрузка токенизатора и QLoRA конфигурации
# ==========================================
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Устанавливаем токен паддинга равным токену конца последовательности (EOS), чтобы избежать ошибок при обработке батчей с разной длиной последовательностей. Это важно для корректного формирования входных данных модели, особенно при обучении и инференсе.

# Конфигурация 4-битного квантования
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

# Загрузка базовой модели в 4-бит
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Подготовка модели к k-bit обучению (заморозка базовых весов, включение градиентов для LayerNorm/Head)
model = prepare_model_for_kbit_training(model)

# ==========================================
# 4. Настройка адаптеров LoRA (PEFT)
# ==========================================
peft_config = LoraConfig(
    r=16,                                 # Ранг матриц адаптации. .Более высокие значения r увеличивают количество обучаемых параметров и могут улучшить качество, но требуют больше памяти и времени на обучение.
    lora_alpha=32,                        # Масштабирующий коэффициент (обычно r * 2). Нужно для масштабирования градиентов адаптеров LoRA, чтобы они были сопоставимы с градиентами базовой модели. Более высокие значения lora_alpha могут ускорить обучение, но также могут привести к переобучению.
    target_modules=[                      # Слои внимания и MLP, куда встраиваются адаптеры
        "q_proj", "k_proj", "v_proj", "o_proj", # Проекции внимания (Query, Key, Value, Output)
        "gate_proj", "up_proj", "down_proj" # Проекции MLP (Feed-Forward Network)
    ],
    lora_dropout=0.05, # Вероятность отключения нейронов в адаптерах LoRA для регуляризации и предотвращения переобучения. Обычно устанавливается в диапазоне 0.05-0.1.
    bias="none", # Тип смещения в адаптерах LoRA. "none" означает отсутствие дополнительных смещений, что уменьшает количество обучаемых параметров и ускоряет обучение.
    task_type="CAUSAL_LM" # Указывает, что задача обучения является задачей автогрессивного языкового моделирования (Causal Language Modeling), где модель предсказывает следующий токен на основе предыдущих токенов в последовательности.
)

model = get_peft_model(model, peft_config) # Интеграция адаптеров LoRA в базовую модель. Функция get_peft_model оборачивает базовую модель, добавляя в неё обучаемые адаптеры LoRA в указанные слои, позволяя эффективно дообучать модель на специфических задачах без изменения исходных весов.
model.print_trainable_parameters()  # Выведет процент обучаемых параметров (~1-2%)

# ==========================================
# 5. Функция форматирования текста (Chat Template)
# ==========================================
def format_prompts(batch):
    texts = []
    for messages in batch["messages"]:
        # Применяем системный шаблонизатор диалогов модели
        formatted_chat = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        texts.append(formatted_chat)
    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)

# ==========================================
# 6. Запуск SFTTrainer (Supervised Fine-Tuning)
# ==========================================
training_args = SFTConfig( # SFTConfig — это конфигурационный класс, который используется для настройки параметров обучения модели с использованием метода Supervised Fine-Tuning (SFT). Он наследует базовые параметры из TrainingArguments библиотеки Transformers и добавляет специфические настройки для SFT.
    output_dir="./qwen-qlora-adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,       # Эффективный батч = 1 * 4 = 4
    learning_rate=2e-4,
    logging_steps=1, # Частота логирования (в шагах). Указывает, как часто будут выводиться метрики обучения и информация о процессе обучения в консоль или лог-файл. Например, logging_steps=1 означает, что информация будет выводиться после каждого шага оптимизации.
    save_total_limit=1,                   # Ограничение на количество сохраняемых контрольных точек
    num_train_epochs=5,                  # Количество эпох
    fp16=not torch.cuda.is_bf16_supported(), # Использование смешанной точности (FP16) для ускорения обучения и уменьшения потребления памяти. Если GPU поддерживает BF16, то fp16=False, иначе fp16=True.
    bf16=torch.cuda.is_bf16_supported(),
    dataset_text_field="text",
    max_seq_length=512,
    save_strategy="epoch" # Стратегия сохранения контрольных точек модели. "epoch" означает, что модель будет сохраняться после каждой завершённой эпохи обучения. Это полезно для восстановления обучения с последней сохранённой точки в случае прерывания процесса или для анализа прогресса модели на разных этапах обучения.
)

trainer = SFTTrainer( # SFTTrainer — это специализированный тренер для обучения моделей с использованием метода Supervised Fine-Tuning (SFT). Он расширяет функциональность стандартного Trainer из библиотеки Transformers, добавляя поддержку специфических задач SFT, таких как обработка диалогов, интеграция LoRA-адаптеров и управление квантованными моделями.
    model=model,
    train_dataset=formatted_dataset,
    # peft_config=peft_config,
    args=training_args, # training_args — это объект конфигурации, который содержит все параметры и настройки для процесса обучения модели. Он определяет такие аспекты, как количество эпох, размер батча, скорость обучения, стратегия сохранения контрольных точек, использование смешанной точности (FP16/BF16), частота логирования и другие важные параметры, влияющие на эффективность и стабильность обучения. Передача training_args в SFTTrainer позволяет тренеру использовать эти настройки для управления процессом обучения модели.
    processing_class=tokenizer # processing_class — это объект токенизатора, который используется для предварительной обработки текстовых данных перед подачей их в модель. В контексте SFTTrainer, токенизатор отвечает за преобразование текстовых последовательностей в числовые представления (токены), которые модель может понимать и обрабатывать. Передача processing_class=tokenizer в SFTTrainer позволяет тренеру автоматически применять токенизацию к входным данным, обеспечивая корректное форматирование и подготовку батчей для обучения модели.
)

print("--- Начинаем обучение ---")
trainer.train()

# Сохраняем веса обученного LoRA-адаптера
model.save_pretrained("./qwen-qlora-adapter")
tokenizer.save_pretrained("./qwen-qlora-adapter")
print("--- Адаптер успешно сохранён в ./qwen-qlora-adapter ---")

In [ ]:
# Инференс с использованием модели, дообученной с помощью QLoRA
from peft import PeftModel

# 1. Загружаем базовую модель заново
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# tokenizer = AutoTokenizer.from_pretrained("./qwen-qlora-adapter")

# 2. Подключаем наш сохранённый LoRA адаптер
fine_tuned_model = PeftModel.from_pretrained(base_model, "./qwen-qlora-adapter")

# 3. Тестовый запрос
messages = [
    {"role": "system", "content": "Ты IT-консультант по архитектуре LLM."},
    {"role": "user", "content": "Зачем нужен QLoRA?"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = fine_tuned_model.generate(**inputs, max_new_tokens=100, temperature=0.3)
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))